# Ankur Task 06: Reproducible Evaluation

This notebook analyzes the frozen Task 05 product using committed sanitized exports. It makes no network or provider call and needs no API key. Human-derived metrics remain pending until R1, R2, and adjudication are complete.

In [1]:
from pathlib import Path
import csv, json, platform, statistics, sys

search_roots = [Path.cwd(), Path('/kaggle/input')]
metric_paths = []
for search_root in search_roots:
    if search_root.exists():
        metric_paths.extend(search_root.rglob('evaluation/exports/aggregate-metrics.json'))
if not metric_paths:
    raise FileNotFoundError('Attach or clone the Ankur repository so evaluation/exports is available.')
METRICS_PATH = sorted(metric_paths, key=lambda value: len(value.parts))[0]
ROOT = METRICS_PATH.parents[2]
def load_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))
metrics = load_json('evaluation/exports/aggregate-metrics.json')
materials = load_json('evaluation/records/public/materials.json')
extraction = load_json('evaluation/records/public/extraction-records.json')
questions = load_json('evaluation/records/public/question-records.json')
written = load_json('evaluation/records/public/written-grading-records.json')
provider = load_json('evaluation/records/public/provider-operations.json')
adaptive = load_json('evaluation/records/public/adaptive-loop-records.json')
baseline = load_json('evaluation/records/public/baseline-records.json')
print('Data root:', ROOT)
print('Loaded:', len(materials), 'materials,', len(questions), 'questions,', len(written), 'written cases')

Data root: E:\WorkBench\hackathon\build-with-gemma-4\ankur
Loaded: 6 materials, 30 questions, 14 written cases


## 1. Project and evaluation objectives

Ankur converts learner-confirmed source material into grounded assessment, grading, revision, and retry artifacts. Task 06 measures corpus coverage, extraction, structural grounding, provider reliability, adaptive completion, and a fair one-prompt baseline. Semantic acceptance and grading agreement require independent human review.

## 2. Corpus summary and provenance/licensing

In [2]:
def counts(rows, field):
    values = {}
    for row in rows:
        values[row[field]] = values.get(row[field], 0) + 1
    return dict(sorted(values.items()))
print('Domains:', counts(materials, 'domain'))
print('Languages:', counts(materials, 'language'))
print('Input types:', counts(materials, 'inputType'))
print('Provenance:', counts(materials, 'provenance'))
print('Licences:', counts(materials, 'licence'))
print('Manual source verification:', counts(materials, 'manualVerificationStatus'))

Domains: {'academic_science': 2, 'bangladesh_civics': 2, 'vocational_safety': 2}
Languages: {'bn': 2, 'en': 2, 'mixed': 2}
Input types: {'digital_pdf': 1, 'mixed_pdf': 1, 'page_image': 2, 'pasted_text': 2}
Provenance: {'team-authored': 6}
Licences: {'CC-BY-4.0': 6}
Manual source verification: {'pending': 6}


## 3. Evaluation protocol

The live plan fixed operation IDs and generation counts before calls, ran sequentially with checkpoint/resume, retained controlled failures, and never regenerated a valid output for preference. Raw provider artifacts stay in ignored private paths. Public records contain sanitized semantic fields, hashes, validation outcomes, and safe operation metadata. R1/R2 review is blinded and adjudicated using the committed guide.

## 4. Extraction metrics

In [3]:
for row in extraction:
    cer = row['characterErrorRate']
    print(row['materialId'], 'p'+str(row['pageNumber']), row['expectedRoute'], 'CER=', 'pending' if cer is None else f'{cer:.4%}', 'changed=', row['changedCharacterCount'])
print('Page success:', metrics['extraction']['pageSuccess'])
print('Routing accuracy:', metrics['extraction']['routingAccuracy'])
print('Mean CER:', metrics['extraction']['meanCharacterErrorRate'])
print('Material-correction heuristic:', metrics['extraction']['materialCorrectionPages'])

SCI-BN-PASTE-01 p1 pasted_text CER= 0.0000% changed= 0
SCI-EN-PDF-01 p1 embedded_text CER= 0.0000% changed= 0
SCI-EN-PDF-01 p2 embedded_text CER= 0.0000% changed= 0
CIV-BN-IMG-01 p1 page_transcription CER= 3.0905% changed= 14
CIV-MIX-PDF-01 p1 embedded_text CER= 0.0000% changed= 0
CIV-MIX-PDF-01 p2 page_transcription CER= 1.0000% changed= 2
CIV-MIX-PDF-01 p3 embedded_text CER= 0.0000% changed= 0
VOC-EN-PASTE-01 p1 pasted_text CER= 0.0000% changed= 0
VOC-MIX-IMG-01 p1 page_transcription CER= 2.0833% changed= 8
Page success: {'count': 9, 'denominator': 9, 'percentage': 100, 'status': 'measured'}
Routing accuracy: {'count': 9, 'denominator': 9, 'percentage': 100, 'status': 'measured'}
Mean CER: 0.00686
Material-correction heuristic: {'count': 2, 'denominator': 9, 'percentage': 22.22, 'status': 'measured'}


## 5. Question-quality metrics

In [4]:
print('Total questions:', metrics['questions']['total'])
print('Deterministic grounding:', metrics['questions']['deterministicGrounding'])
print('Deterministic key invariant:', metrics['questions']['deterministicKeyValidity'])
print('Potential cross-run duplicates:', metrics['questions']['duplicates'])
print('Human acceptance:', metrics['questions']['humanAccepted'])
print('Human grounding/key/ambiguity:', metrics['questions']['humanGroundedAccepted'], metrics['questions']['humanCorrectKeys'], metrics['questions']['humanAmbiguous'])

Total questions: 30
Deterministic grounding: {'count': 30, 'denominator': 30, 'percentage': 100, 'status': 'measured'}
Deterministic key invariant: {'count': 30, 'denominator': 30, 'percentage': 100, 'status': 'measured'}
Potential cross-run duplicates: {'count': 2, 'denominator': 30, 'percentage': 6.67, 'status': 'measured'}
Human acceptance: {'count': 0, 'denominator': 0, 'percentage': None, 'status': 'pending_human_review'}
Human grounding/key/ambiguity: {'count': 0, 'denominator': 0, 'percentage': None, 'status': 'pending_human_review'} {'count': 0, 'denominator': 0, 'percentage': None, 'status': 'pending_human_review'} {'count': 0, 'denominator': 0, 'percentage': None, 'status': 'pending_human_review'}


## 6. Written-grading metrics

In [5]:
print('Answer-case coverage:', counts(written, 'answerCase'))
print('Provider-eligible cases:', metrics['written']['providerOperations'])
print('Available mark/status pairs:')
for row in written:
    print(' ', row['recordId'], row['awardedMarks'], row['status'])
print('Human reviewed:', metrics['written']['humanReviewed'])
print('MAE / agreement:', metrics['written']['meanAbsoluteError'], metrics['written']['exactAgreement'], metrics['written']['withinOneMark'], metrics['written']['statusAgreement'])

Answer-case coverage: {'correct': 6, 'empty': 2, 'incorrect': 2, 'missing_key_concept': 1, 'partially_correct': 2, 'unsupported_claim': 1}
Provider-eligible cases: 12
Available mark/status pairs:
  written-record:SCI-BN-PASTE-01:correct 5 correct
  written-record:SCI-BN-PASTE-01:partially_correct 3 partially_correct
  written-record:SCI-BN-PASTE-01:retry-correct 5 correct
  written-record:SCI-EN-PDF-01:incorrect 0 incorrect
  written-record:SCI-EN-PDF-01:empty 0 not_answered
  written-record:SCI-EN-PDF-01:retry-correct 5 correct
  written-record:CIV-MIX-PDF-01:correct 5 correct
  written-record:CIV-MIX-PDF-01:partially_correct 4 partially_correct
  written-record:VOC-EN-PASTE-01:incorrect 0 incorrect
  written-record:VOC-EN-PASTE-01:empty 0 not_answered
  written-record:VOC-EN-PASTE-01:retry-correct 5 correct
  written-record:VOC-MIX-IMG-01:unsupported_claim 0 incorrect
  written-record:VOC-MIX-IMG-01:missing_key_concept 0 incorrect
  written-record:VOC-MIX-IMG-01:retry-correct None pe

## 7. Reliability and latency

In [6]:
reliability = metrics['reliability']
for key in ['totalOperations','firstPassValid','finalValid','repairRate','repairSuccess','groundingFailures','quoteFailures','conceptFailures','reconciliationFailures','medianLatencyMs','p95LatencyMs','maximumLatencyMs']:
    print(key, reliability[key])
failures = [row for row in provider if row['finalStatus'] == 'controlled_failure']
print('Controlled failures by category:', counts(failures, 'failureCategory'))
print('Controlled failures by operation:', counts(failures, 'operationType'))

totalOperations 51
firstPassValid {'count': 24, 'denominator': 51, 'percentage': 47.06, 'status': 'measured'}
finalValid {'count': 40, 'denominator': 51, 'percentage': 78.43, 'status': 'measured'}
repairRate {'count': 23, 'denominator': 51, 'percentage': 45.1, 'status': 'measured'}
repairSuccess {'count': 16, 'denominator': 23, 'percentage': 69.57, 'status': 'measured'}
groundingFailures 0
quoteFailures 0
conceptFailures 0
reconciliationFailures 0
medianLatencyMs 26252
p95LatencyMs 93741
maximumLatencyMs 190269
Controlled failures by category: {'EVIDENCE_INVALID': 2, 'INVALID_OUTPUT': 5, 'RATE_LIMITED': 1, 'TIMEOUT': 2, 'UNAVAILABLE': 1}
Controlled failures by operation: {'analysis': 4, 'assessment_generation': 3, 'revision_retry_generation': 2, 'written_grading': 2}


## 8. Adaptive-loop metrics

In [7]:
print('Adaptive aggregate:', metrics['adaptive'])
for row in adaptive:
    print(row['materialId'], row['status'], row['revisionMode'], row['originalScore'], row['retryScore'], row['scoreChange'], row['failureCategory'])

Adaptive aggregate: {'total': 6, 'valid': {'count': 3, 'denominator': 6, 'percentage': 50, 'status': 'measured'}, 'fabricatedWeaknesses': 0, 'meanObservedScoreChange': 5}
SCI-BN-PASTE-01 valid reinforcement 3 6 3 None
SCI-EN-PDF-01 valid weak_area 0 6 6 None
CIV-BN-IMG-01 controlled_failure None None None None EVIDENCE_INVALID
CIV-MIX-PDF-01 controlled_failure None None None None INVALID_OUTPUT
VOC-EN-PASTE-01 valid weak_area 0 6 6 None
VOC-MIX-IMG-01 controlled_failure weak_area None None None TIMEOUT


## 9. Structured Ankur versus one-prompt baseline

In [8]:
print('Structured questions:', len(questions), 'grounding-valid:', sum(row['deterministicGroundingValid'] for row in questions))
print('Baseline materials:', len(baseline), 'parsed questions:', sum(row['parsedQuestionCount'] for row in baseline))
print('Baseline parse success:', metrics['baseline']['parseSuccess'])
print('Baseline evidence-labelled lines:', metrics['baseline']['evidenceTransparency'])
print('Human quality comparison: pending for both systems')

Structured questions: 30 grounding-valid: 30
Baseline materials: 6 parsed questions: 30
Baseline parse success: {'count': 6, 'denominator': 6, 'percentage': 100, 'status': 'measured'}
Baseline evidence-labelled lines: {'count': 30, 'denominator': 30, 'percentage': 100, 'status': 'measured'}
Human quality comparison: pending for both systems


## 10. Per-language and per-domain breakdowns

In [9]:
def question_breakdown(field):
    result = {}
    for row in questions:
        bucket = result.setdefault(row[field], {'questions':0,'grounded':0,'duplicates':0})
        bucket['questions'] += 1
        bucket['grounded'] += int(row['deterministicGroundingValid'])
        bucket['duplicates'] += int(row['duplicateOfRecordId'] is not None)
    return dict(sorted(result.items()))
print('Language:', question_breakdown('language'))
print('Domain:', question_breakdown('domain'))
print('Stage:', counts(questions, 'questionStage'))

Language: {'bn': {'questions': 8, 'grounded': 8, 'duplicates': 1}, 'en': {'questions': 12, 'grounded': 12, 'duplicates': 1}, 'mixed': {'questions': 10, 'grounded': 10, 'duplicates': 0}}
Domain: {'academic_science': {'questions': 16, 'grounded': 16, 'duplicates': 2}, 'bangladesh_civics': {'questions': 4, 'grounded': 4, 'duplicates': 0}, 'vocational_safety': {'questions': 10, 'grounded': 10, 'duplicates': 0}}
Stage: {'adaptive_retry': 8, 'original_assessment': 22}


## 11. Error analysis

In [10]:
for row in failures:
    print(row['operationId'], row['operationType'], row['failureCategory'], str(row['latencyMs'])+'ms', 'repair='+str(row['repairAttempted']))
print('Accepted-artifact grounding/quote/concept/reconciliation failures:', reliability['groundingFailures'], reliability['quoteFailures'], reliability['conceptFailures'], reliability['reconciliationFailures'])

analysis:CIV-BN-IMG-01 analysis EVIDENCE_INVALID 26252ms repair=True
analysis:CIV-BN-IMG-01:attempt1 analysis RATE_LIMITED 55729ms repair=False
analysis:VOC-EN-PASTE-01:attempt1 analysis UNAVAILABLE 5797ms repair=False
analysis:VOC-MIX-IMG-01:attempt1 analysis EVIDENCE_INVALID 24884ms repair=True
assessment:CIV-MIX-PDF-01:r1:attempt1 assessment_generation INVALID_OUTPUT 49254ms repair=True
assessment:VOC-EN-PASTE-01:r1 assessment_generation INVALID_OUTPUT 51241ms repair=True
assessment:VOC-EN-PASTE-01:r1:attempt1 assessment_generation INVALID_OUTPUT 62391ms repair=True
retry-written:VOC-MIX-IMG-01:correct written_grading TIMEOUT 55010ms repair=False
revision:CIV-MIX-PDF-01 revision_retry_generation INVALID_OUTPUT 75495ms repair=True
revision:CIV-MIX-PDF-01:attempt1 revision_retry_generation INVALID_OUTPUT 75112ms repair=True
written:SCI-EN-PDF-01:incorrect:attempt1 written_grading TIMEOUT 55021ms repair=False
Accepted-artifact grounding/quote/concept/reconciliation failures: 0 0 0 0


## 12. Public-safe figures

In [11]:
FIGURES = ROOT / 'evaluation/notebook/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
def bar_svg(path, title, labels, values, maximum=100):
    width, height = 820, 110 + 58*len(labels)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="#fffdf5"/>', f'<text x="28" y="42" font-family="system-ui" font-size="24" fill="#0d5d43">{title}</text>']
    for index, (label, value) in enumerate(zip(labels, values)):
        y = 76 + index*58
        bar = 560 * value / maximum if maximum else 0
        parts += [f'<text x="28" y="{y+22}" font-family="system-ui" font-size="16" fill="#173c31">{label}</text>', f'<rect x="220" y="{y}" width="560" height="28" rx="8" fill="#e5eadf"/>', f'<rect x="220" y="{y}" width="{bar:.1f}" height="28" rx="8" fill="#2f8a62"/>', f'<text x="230" y="{y+20}" font-family="system-ui" font-size="14" fill="#173c31">{value:.2f}%</text>']
    parts.append('</svg>')
    path.write_text(''.join(parts), encoding='utf-8')
bar_svg(FIGURES/'reliability.svg', 'Provider reliability', ['First pass valid','Final valid','Repair success'], [reliability['firstPassValid']['percentage'], reliability['finalValid']['percentage'], reliability['repairSuccess']['percentage']])
bar_svg(FIGURES/'coverage.svg', 'Automated evaluation coverage', ['Extraction pages','Question grounding','Written records','Adaptive valid'], [metrics['extraction']['pageSuccess']['percentage'], metrics['questions']['deterministicGrounding']['percentage'], 100.0, metrics['adaptive']['valid']['percentage']])
print('Wrote:', FIGURES/'reliability.svg', 'and', FIGURES/'coverage.svg')

Wrote: E:\WorkBench\hackathon\build-with-gemma-4\ankur\evaluation\notebook\figures\reliability.svg and E:\WorkBench\hackathon\build-with-gemma-4\ankur\evaluation\notebook\figures\coverage.svg


## 13. Limitations

- Human source verification, question acceptance, answer-key correctness, ambiguity, written-score agreement, and feedback usefulness are pending.
- Deterministic grounding validates IDs and quotations, not pedagogical correctness.
- The small team-authored corpus cannot represent all layouts, dialects, domains, or provider conditions.
- Immediate retry changes do not demonstrate durable learning.
- The one-prompt baseline is a simple documented comparison, not a claim against every alternative design.

## 14. Reproducibility and data dictionary

In [12]:
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Dependencies: Python standard library only; no network or API key')
print('Authoritative data dictionary:', ROOT/'evaluation/DATA_DICTIONARY.md')
print('Metric schema:', metrics['schemaVersion'])
print('Record schema versions:', sorted({row['schemaVersion'] for row in questions + written + provider + adaptive + baseline}))

Python: 3.11.9
Platform: Windows-10-10.0.26200-SP0
Dependencies: Python standard library only; no network or API key
Authoritative data dictionary: E:\WorkBench\hackathon\build-with-gemma-4\ankur\evaluation\DATA_DICTIONARY.md
Metric schema: aggregate-metrics.v1
Record schema versions: ['adaptive-loop-record.v1', 'baseline-record.v1', 'generated-question-record.v1', 'provider-operation.v1', 'written-grading-record.v1']


## 15. Conclusions

The measured package meets the corpus, 30-question, 12-written-case, six-adaptive-record, and six-material baseline scale. Accepted structured artifacts had zero recorded grounding, quotation, concept-reference, or mark-reconciliation failures, while final-valid provider reliability was below the 95% target. Human quality and grading agreement cannot be concluded until the prepared two-reviewer protocol is completed.